Refer to https://github.com/GreenleafLab/HDMA/blob/main/code/05-misc/04-ChromBPNet_use_cases.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

# deal with matplotlib plots
%matplotlib inline

# display all outputs in a cell
get_ipython().ast_node_interactivity = 'all'

from IPython.display import display, HTML

In [ ]:
# misc
import sys
import os
import numpy as np
import random
from tqdm import tqdm
import pandas as pd

# io
import h5py
from pyfaidx import Fasta

# ML
import bpnetlite
from bpnetlite import ChromBPNet
from bpnetlite import BPNet
from bpnetlite.bpnet import CountWrapper
from bpnetlite.attribute import deep_lift_shap

import torch

import pickle

import pyranges as pr

In [ ]:
# plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn; seaborn.set_style('white')
import matplotlib.ticker as ticker
import plotnine as pn
import logomaker
import tangermeme
from tangermeme.plot import plot_logo

from tangermeme.utils import random_one_hot, _validate_input
from tangermeme.utils import characters
from tangermeme.ersatz import substitute
from tangermeme.utils import one_hot_encode

print(np.__version__)
print(tangermeme.__version__)
print(bpnetlite.__version__)
print(torch.__version__)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES']='0'
#os.environ['CUDA_VISIBLE_DEVICES']='MIG-40f43250-998e-586a-ac37-d6520e92590f'

torch.manual_seed(100)

# editable text in PDFs
# https://stackoverflow.com/a/54111532
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sys.path.append("~/HDMA/code/03-chrombpnet")
import tangermeme_utils as tanutils
from tangermeme_utils.utils import plot_motif
from tangermeme_utils.eval import summarize_across_folds, mean_across_sequences
from tangermeme_utils.wrappers import ChromBPNetWrapper, reshape_folds

In [ ]:
torch.cuda.device_count()

In [ ]:
dev = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dev

torch.cuda.is_available()

t1 = torch.randn(1,2).to(dev)
t1
t1.device

In [ ]:
INPUTLEN = 2114
OUTPUTLEN = 1000
SHIFT_REL_INPUT = np.int64((2114 - 1000) / 2)
SHIFT_REL_INPUT

NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]

LOGO_ALPHABET = 'ACGT'

# finemo colorscheme
LOGO_COLORS = {"A": '#109648', "C": '#255C99', "G": '#F7B32B', "T": '#D62839'}

# logomaker colorscheme
LOGO_COLORS2= {
        'G': [1, .65, 0],
        'T': [1, 0, 0],
        'C': [0, 0, 1],
        'A': [0, .5, 0]
    }

CHAR_IGNORE = ['QWERYUIOPSDFHJKLZXVBNM']

In [ ]:
genome_fa = os.path.join("~/software/ref_tree_shrew/TreeShrew_scATAC_ref/fasta/genome.fa")

In [ ]:
tup_KIZ2021 = Fasta(genome_fa)

In [ ]:
def predict_sequence(chrom,start,end,genome,subtype,models,models_counts,min_pos=557,max_pos=1557,show_fig=True):

    mid=(end+start)//2

    input_start=mid - INPUTLEN//2
    input_end=mid + INPUTLEN//2
    pre_seq = str(genome[chrom][mid - INPUTLEN//2 : mid + INPUTLEN//2])
    assert len(pre_seq) == 2114
    output_start=mid - OUTPUTLEN//2
    output_end=mid + OUTPUTLEN//2
    

    #使用 .upper() 转换为大写
    seq_ohe = one_hot_encode(pre_seq.upper()).type(torch.float32).unsqueeze(0) #seq_ohe.shape，torch.Size([1, 4, 2114])。1，batch size,一次输入一条序列；4，四个碱基通道；2114，序列长度。
    # predict
    print("@ predicting...")
    seq_pred_list = []

    for model in models:
        y_seq = tangermeme.predict.predict(model, seq_ohe)
        seq_pred_list.append(y_seq)

    seq_pred = reshape_folds(seq_pred_list)

    seq_pred_profile_mean, _ = summarize_across_folds(seq_pred[0])

    seq_pred_profile_mean.shape

    # compute attribution
    print("@ computing attribution...")
    try:

        seq_attr_list = []
        
        for model in models_counts:
            y_seq = deep_lift_shap(model, seq_ohe, random_state=0)#torch.Size([1, 4, 2114])
            seq_attr_list.append(y_seq)

        seq_attr = torch.stack(seq_attr_list, dim=0)#torch.Size([5, 1, 4, 2114])。5，5个fold。
        seq_attr_mean, _ = summarize_across_folds(seq_attr)#5 个 fold 求平均。torch.Size([1, 4, 2114])。

        
    except Exception as e:
        print(f"Error during attribution computation: {e}")
        raise #raise 的作用是：把刚刚捕获到的异常重新抛出，让程序立即停止，并显示完整的错误追踪信息

    results={"seq_pred_profile_mean":seq_pred_profile_mean,
        "seq_attr_mean":seq_attr_mean}

    pred_seq = results['seq_pred_profile_mean'].squeeze()

    attr_seq = results['seq_attr_mean'].sum(axis = 1).squeeze().T[(INPUTLEN // 2 - 500):(INPUTLEN // 2 + 500)]

    #输出维度信息
    print("seq_ohe shape:", seq_ohe.shape) #torch.Size([1, 4, 2114])
    print("single-fold profile shape:", seq_pred_list[0][0].shape) #torch.Size([1, 1, 1000])
    print("single-fold attribution shape:", seq_attr_list[0].shape) #torch.Size([1, 4, 2114])
    print("all-fold profile shape:", seq_pred[0].shape) #torch.Size([5, 1, 1, 1000])
    print("all-fold attribution shape:", seq_attr.shape) #torch.Size([5, 1, 4, 2114])
    print("mean profile shape:", seq_pred_profile_mean.shape) #torch.Size([1, 1, 1000])
    print("mean attribution shape:", seq_attr_mean.shape) #torch.Size([1, 4, 2114])
    print("pred_seq shape:", pred_seq.shape) #torch.Size([1000])
    print("attr_seq shape:", attr_seq.shape) #torch.Size([1000])

    # ploting-1
    fig, ax = plt.subplots(3, 1, figsize = (25, 6))
    alpha = 0.3
    # prediction in 1,000 bp
    ax[0].plot(pred_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[0].set_ylabel("Pred. \naccessibility")
    ax[0].axvspan(min_pos-557, max_pos-557, alpha = alpha, color='yellow')
    ax[0].legend(fontsize = 8)
    # attribution in 1,000 bp
    ax[1].plot(attr_seq, linewidth = 0.5, label = "seq(5-fold mean)", color = "blue")
    ax[1].set_ylabel("Contrib. \nscores")
    ax[1].axvspan(min_pos-557, max_pos-557, alpha = alpha, color = 'yellow')
    ax[1].legend(fontsize = 8)
    #
    ylim = None
    tangermeme.plot.plot_logo(np.float64(results['seq_attr_mean'][0]), start = min_pos, end = max_pos, ax=ax[2], ylim = ylim) #results['seq_attr_mean'][0].shape,torch.Size([4, 2114])，画logo是传入，根据四个碱基通道的贡献值 #end 是开区间，不包含 end 本身。
    ax[2].set_ylabel("Contrib. \n")
        # put a title at the very top
    #plt.suptitle(subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end), fontsize = 12)
    plt.suptitle(subtype + ":" + chrom + ":"+ str(output_start+1) + "-" + str(output_end), fontsize = 12)
    plt.xlabel("Genomic position (bp)")
    plt.tight_layout()

    # ============================================================
    # 输出 contribution 区间和 prediction track 区间的基因组坐标
    # ============================================================

    # ----------------------------
    # 1. Contribution (2114 bp input)
    # ----------------------------

    contribution_genome_start = input_start + min_pos
    contribution_genome_end   = input_start + max_pos

    print(
        f"Contribution region (input 2114bp): "
        f"{chrom}:{contribution_genome_start+1}-{contribution_genome_end}"
    )


    # ----------------------------
    # 2. Prediction track (1000 bp output)
    # ----------------------------

    output_offset = (INPUTLEN - OUTPUTLEN)//2   # =557

    pred_region_start_idx = min_pos - output_offset
    pred_region_end_idx   = max_pos - output_offset


    # 防止超出prediction范围
    pred_region_start_idx = max(0, pred_region_start_idx)
    pred_region_end_idx = min(OUTPUTLEN, pred_region_end_idx)


    prediction_genome_start = output_start + pred_region_start_idx
    prediction_genome_end   = output_start + pred_region_end_idx


    print(
        f"Prediction region (output 1000bp): "
        f"{chrom}:{prediction_genome_start+1}-{prediction_genome_end}"
    )

    # ============================================================
    # 计算与 plot_logo 完全一致区间内的 contribution score，得到最大值和最小值
    # ============================================================

    contribution_region=results['seq_attr_mean'].sum(axis = 1).squeeze().T[min_pos:max_pos] #python切片规则，左闭右开 [start, end)
    # 最大值和最小值
    contribution_max = contribution_region.max().item()
    contribution_min = contribution_region.min().item()

    print("Maximum contribution score:", contribution_max)
    print("Minimum contribution score:", contribution_min)
    
    if show_fig:
        plt.show()
    else:
        #plt.savefig(os.path.join(outdir, subtype + ":" + chrom + ":"+ str(start+1) + "-" + str(end) + ".pdf"))
        plt.savefig(os.path.join(outdir, subtype + ":" + chrom + ":"+ str(output_start+1) + "-" + str(output_end) + ".pdf"))

    plt.close(fig)
    
    '''
    ## ploting-2
    import logomaker
    attr_data = results['seq_attr_mean'][0].detach().cpu().numpy()
    attr_data_T = attr_data.T
    df = pd.DataFrame(attr_data_T, columns=['A', 'C', 'G', 'T'])
    crp_logo = logomaker.Logo(df, 
                          figsize=(120, 6), # 2114bp 建议先设宽一点
                          shade_below=0,
                          fade_below=0)
    ### ploting-3
    min_pos=min_pos
    max_pos=max_pos
    df_trimmed = df.loc[min_pos:max_pos,:]
    crp_logo = logomaker.Logo(df_trimmed, 
                            figsize=(25, 2), # 2114bp 建议先设宽一点
                            shade_below=0,
                            fade_below=0)
'''
    

In [ ]:
outdir="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/marker_peak_for_validation/plot_400bp_track"

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C3.2/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C3.2/models/chrombpnet_nobias.h5")))
]

In [ ]:
#1.chr5:30668503-30669503
chrom="chr5"
start=30668503
end=30669503
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=845,max_pos=1245,show_fig=True) #288+30668503,30669503-312 #chr5:30668792-30669191

In [ ]:
#2.chr5:30734960-30735960
chrom="chr5"
start=30734960
end=30735960
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=890,max_pos=1290,show_fig=True)#333+30734960,30735960-267 #chr5:30735294-30735693

In [ ]:
#3.chr8:38991253-38992253
chrom="chr8"
start=38991253
end=38992253
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=760,max_pos=1160,show_fig=True) #203+38991253,38992253-397 #chr8:38991457-38991856

In [ ]:
#4.chr8:53216905-53217905
chrom="chr8"
start=53216905
end=53217905
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=200+557,max_pos=600+557,show_fig=True) #200+53216905,600+53216905 #chr8:53217106-53217505

In [ ]:
#5.chr9:63918357-63919357
chrom="chr9"
start=63918357
end=63919357
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=900,max_pos=1300,show_fig=True) #343+63918357,63919357-257 #chr9:63918701-63919100

In [ ]:
#6.chr15:47582755-47583755
chrom="chr15"
start=47582755
end=47583755
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=770,max_pos=1170,show_fig=True) #213+47582755,47583755-387 #chr15:47582969-47583368

In [ ]:
#7.chr16:57254798-57255798
chrom="chr16"
start=57254798
end=57255798
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=935,max_pos=1335,show_fig=True) #378+57254798,57255798-222 #chr16:57255177-57255576

In [ ]:
#8.chr17:1717547-1718547
chrom="chr17"
start=1717547
end=1718547
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=850,max_pos=1250,show_fig=True) #293+1717547,1718547-307 #chr17:1717841-1718240

In [ ]:
#9.chr17:37609343-37610343
chrom="chr17"
start=37609343
end=37610343
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=950,max_pos=1350,show_fig=True) #393+37609343,37610343-207 #chr17:37609737-37610136

In [ ]:
#10.chr19:1079746-1080746
chrom="chr19"
start=1079746
end=1080746
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=850,max_pos=1250,show_fig=True) #293+1079746,1080746-307 #chr19:1080040-1080439

In [ ]:
outdir="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/predict/C3.2/marker_peak_for_validation/plot_600bp_track"

In [ ]:
proj_in="~/chrombpnet/chrombpnet_use_snakefile/Treeshrew/TS_C3Cardiomyocytes/output/models"

In [ ]:
#load model
models = [
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C3.2/models/chrombpnet_nobias.h5"))),
    ChromBPNetWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C3.2/models/chrombpnet_nobias.h5")))
]

models_counts = [
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_0/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_1/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_2/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_3/C3.2/models/chrombpnet_nobias.h5"))),
    CountWrapper(BPNet.from_chrombpnet(filename=os.path.join(proj_in, f"fold_4/C3.2/models/chrombpnet_nobias.h5")))
]

In [ ]:
#1.chr5:30668503-30669503
chrom="chr5"
start=30668503
end=30669503
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+200,max_pos=1557-200,show_fig=True) #chr5:30668704-30669303

In [ ]:
#2.chr5:30734960-30735960
chrom="chr5"
start=30734960
end=30735960
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+150,max_pos=1557-250,show_fig=True) #chr5:30735111-30735710

In [ ]:
#3.chr8:38991253-38992253
chrom="chr8"
start=38991253
end=38992253
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+200,max_pos=1557-200,show_fig=True) #chr8:38991454-38992053

In [ ]:
#4.chr8:53216905-53217905
chrom="chr8"
start=53216905
end=53217905
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+100,max_pos=1557-300,show_fig=True) #chr8:53217006-53217605

In [ ]:
#5.chr9:63918357-63919357
chrom="chr9"
start=63918357
end=63919357
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+200,max_pos=1557-200,show_fig=True) #chr9:63918558-63919157

In [ ]:
#6.chr15:47582755-47583755
chrom="chr15"
start=47582755
end=47583755
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+150,max_pos=1557-250,show_fig=True) #chr15:47582906-47583505

In [ ]:
#7.chr16:57254798-57255798
chrom="chr16"
start=57254798
end=57255798
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+300,max_pos=1557-100,show_fig=True) #chr16:57255099-57255698

In [ ]:
#8.chr17:1717547-1718547
chrom="chr17"
start=1717547
end=1718547
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+200,max_pos=1557-200,show_fig=True) #chr17:1717748-1718347

In [ ]:
#9.chr17:37609343-37610343
chrom="chr17"
start=37609343
end=37610343
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+300,max_pos=1557-100,show_fig=True) #chr17:37609644-37610243

In [ ]:
#10.chr19:1079746-1080746
chrom="chr19"
start=1079746
end=1080746
#predict_sequence(chrom,start,end,subtype,models,models_counts,show_fig=True)
#predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557,max_pos=1557,show_fig=True)
predict_sequence(chrom,start,end,tup_KIZ2021,subtype="C3.2",models=models,models_counts=models_counts,min_pos=557+200,max_pos=1557-200,show_fig=True) #chr19:1079947-1080546